# MV-BrainMAE — RUN-ALL notebook (Kaggle GPU)

Runs the **whole pipeline idempotently**: every stage checks whether its output
already exists and skips it, and pretraining stops itself before the 12 h
session limit and **resumes on the next run**. So the workflow is simply:

1. Attach your private `oasis1` dataset (see repo `docs/KAGGLE_SETUP.md`), set GPU on.
2. **Run All.** When the session ends or the budget runs out, save a version and
   press **Run All again** — completed stages skip, pretraining continues.
3. After 2–4 runs everything is finished; the last cell zips all artifacts.

Stage times (T4×2): factory ~20 min · pretrain single 3–5 h · pretrain cross
5–8 h · downstream 1.5–4 h · anomaly ~30 min · analysis <5 min.

In [ ]:
import os, sys, subprocess, time
from pathlib import Path

IN_KAGGLE = Path("/kaggle").is_dir()
REPO_URL = os.environ.get("REPO_URL",
    "https://github.com/YOUR_GITHUB_USERNAME/mv-brainmae.git")  # <- set after pushing
if IN_KAGGLE:
    repo = Path("mv-brainmae")
    if not repo.is_dir():
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL], check=True)
else:
    repo = Path("..")
sys.path.insert(0, str(repo.resolve()))
os.chdir(repo)
import torch
print("repo:", repo.resolve(), "| torch", torch.__version__,
      "| device:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

In [ ]:
# ---- session budget & idempotent stage gates ----
SESSION_MINUTES = 540          # keep < 720 (Kaggle kills at 12h); guard stops earlier
FORCE = set()                  # e.g. {"downstream"} to rerun a completed stage
T0 = time.time()
def remaining():  return max(5.0, SESSION_MINUTES - (time.time() - T0) / 60)
def elapsed():    return f"{(time.time() - T0) / 60:.0f} min"

def stage_done(kind, path):
    p = Path(path)
    if kind in FORCE or not p.exists():
        return False
    if kind == "pretrain":
        import json
        s = json.loads(p.read_text())
        return s.get("stop_reason") == "completed"
    return True

print(f"session budget {SESSION_MINUTES} min; stages skip automatically when done")

## Stage 1 — data factory (volumes → slices + subject-level splits)

In [ ]:
from mvbrainmae.utils import load_config
if stage_done("file", "data/factory/metadata.csv"):
    print("STAGE 1 already done — skipping (add 'factory' to FORCE to redo)")
else:
    from mvbrainmae.factory import run_factory
    cfg = load_config("configs/data_factory.yaml")
    if IN_KAGGLE:
        cfg["root"] = "/kaggle/input/oasis1"      # <- your attached dataset id
    print(run_factory(cfg))

## Stage 2 — pretraining: 2A single-view MAE, then 2B multi-view cross-plane MAE

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.pretrain import run_pretrain
for mode in ("single", "cross"):
    if stage_done("pretrain", f"outputs/pretrain_{mode}/pretrain_summary.json"):
        print(f"STAGE 2-{mode} already COMPLETED — skipping")
        continue
    cfg = load_config(f"configs/pretrain_{mode}.yaml")
    cfg["max_minutes"] = min(float(cfg.get("max_minutes", 600)), remaining() - 15)
    if cfg["max_minutes"] < 20:
        print(f"not enough session time left for pretrain_{mode} — rerun notebook later")
        break
    print(run_pretrain(cfg))   # resumable: next notebook version continues

## Stage 3 — downstream probes / label efficiency / finetune / age probe

In [ ]:
from mvbrainmae.utils import load_config
if stage_done("file", "results/downstream_summary.json"):
    print("STAGE 3 already done — skipping")
else:
    from mvbrainmae.downstream import run_downstream
    print(run_downstream(load_config("configs/downstream.yaml")))

## Stage 4 — reconstruction-error anomaly maps

In [ ]:
from mvbrainmae.utils import load_config
if stage_done("file", "results/anomaly_metrics.json"):
    print("STAGE 4 already done — skipping")
else:
    from mvbrainmae.anomaly import run_anomaly
    print(run_anomaly(load_config("configs/anomaly.yaml")))

## Stage 5 — figures + auto hypothesis verdicts

In [ ]:
from mvbrainmae.utils import load_config
from mvbrainmae.analysis import run_analysis
print(run_analysis(load_config("configs/analysis.yaml")))
print(Path("results/summary.md").read_text())

## Artifacts (download this zip afterwards)

In [ ]:
import os, zipfile
out_zip = "/kaggle/working/mv_brainmae_artifacts.zip" if IN_KAGGLE else "mv_brainmae_artifacts.zip"
with zipfile.ZipFile(out_zip, "w", zipfile.ZIP_DEFLATED) as z:
    for d in ("results", "figures"):
        for root, _, files in os.walk(d):
            for fn in files:
                z.write(os.path.join(root, fn))
    for p in Path("outputs").glob("*/encoder.pt"): z.write(p)
    for p in Path("outputs").glob("*/pretrain_*.jsonl"): z.write(p)
    for p in Path("data/factory").glob("*.csv"): z.write(p)
    z.write("data/factory/stats.json")
print("ARTIFACTS ->", out_zip, "| total elapsed:", elapsed())